# 08 — Rebuild the evidence report from artifacts

## Question

Can every reported result and limitation be traced to retained inputs, saved predictions, receipts and gate decisions?

## Inputs

Evaluation, adaptation, optional and development-snapshot receipts from this run. No notebook state from earlier kernels is required.

Use an explicit `STV2_CONFIG` JSON file and a unique run ID. The setup resolves relative artifact paths from the repository root. See the [execution guide](README.md), [development protocol](../../docs/studies/synthetic-training-v2/protocol.md) and [literature ledger](../../docs/studies/synthetic-training-v2/literature.md).

In [ ]:
import os
import sys
from pathlib import Path
from IPython.display import Image, display

if not os.environ.get("STV2_CONFIG"):
    raise RuntimeError("Set STV2_CONFIG to an explicit study JSON configuration before execution.")
config_path = Path(os.environ["STV2_CONFIG"]).expanduser().resolve()
if not config_path.is_file():
    raise FileNotFoundError(f"Study configuration does not exist: {config_path}")
search_root = Path(os.environ.get("GAVD6_ROOT", Path.cwd())).expanduser().resolve()
PROJECT_ROOT = next((path for path in (search_root, *search_root.parents)
                     if (path / "src/gavd6_sjepa").is_dir() and (path / "pyproject.toml").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run inside the repository or set GAVD6_ROOT to its root.")
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
os.environ["STV2_CONFIG"] = str(config_path)
from gavd6_sjepa.research_directions.synthetic_training_v2.config import RunConfig
from gavd6_sjepa.research_directions.synthetic_training_v2.workflow import run_stage

cfg = RunConfig.load(os.environ["STV2_CONFIG"])
display({"run_id": cfg.run_id, "mode": cfg.mode, "device": cfg.device,
         "artifact_root": str(cfg.root), "confirmation": "closed"})
if cfg.mode == "fixture":
    print("CPU software fixture: method ordering is not empirical evidence.")

## Computation

Reconstruct the report from saved per-window predictions and group records. Include branch decisions, evidence origin, cost limits and the strongest competing explanation.

`run_stage` implements the computation in the study modules. It checks prerequisite receipts and returns the saved result on an unchanged rerun; a changed configuration or code identity requires a new run ID.

In [ ]:
report = run_stage(cfg, "report", repo_root=PROJECT_ROOT)
display({"report": report["report"], "evidence_status": report["evidence_status"]})
display(report["gates"])

## Outputs and checks

Open the reported `report.md` path alongside `identity.json`, `effective-config.json`, `environment.json`, stage receipts and per-fit cost records. Full tables remain in artifacts rather than the canonical notebook. Independently compare the generated report against saved predictions and unresolved review findings.

Stage receipts under `receipts/` record elapsed time and hashes of produced artifacts. Inspect the saved files for full diagnostics; the display above is deliberately brief.

## Interpretation

A fixture report is executable software evidence, not synthetic-to-real validation or a confirmation result. The narrow hypotheses concern motion-preserving extractor-shift evaluation and paired JEPA's added value over matched coordinate learning. Novelty depends on an empirical result beyond the closest prior work. Temporal refinement, synthetic pairing and masked pose reconstruction already exist.

A completed fixture checks software behavior. Scientific gates use `pass`, `fail` or `insufficient_evidence`; fixture success cannot make a scientific gate pass.

## Next gate

Independent review must inspect implementation and saved artifacts, then record dispositions. Source data, runtime, anatomical references, real temporal annotations, repeated finalists and calibrated margins remain separate empirical prerequisites. See the [protocol review](../../docs/studies/synthetic-training-v2/protocol-review.md).